# The C Language — A Complete Hands-On Course in Google Colab

This notebook walks through the whole C language, from your first `printf` to multi-file projects with a Makefile. Every topic has short notes, a real program, and a cell that compiles and runs it.

## How this notebook works

Colab runs on a Linux machine that already has **GCC** (the GNU C Compiler) installed. Each lesson uses two kinds of cells:

- A cell starting with `%%writefile path/file.c` **saves** the code below it as a file. It does not run anything.
- A cell starting with `!` sends a command to the Linux terminal. We use it to **compile** with `gcc` and **run** the program.

Every compile uses these flags, and they're worth getting used to:

| Flag | Meaning |
|---|---|
| `-Wall -Wextra` | turn on warnings: the compiler points out likely bugs |
| `-std=c11` | use the C11 standard |
| `-o name` | name of the program to produce |

## File layout created by this notebook

```
/content/c_course/
├── 01_basics/        hello, types, operators, input
├── 02_control/       if/else, switch, loops
├── 03_functions/     functions, recursion, scope
├── 04_arrays/        arrays and strings
├── 05_pointers/      pointers and dynamic memory
├── 06_structs/       struct, union, enum, typedef
├── 07_files/         file input/output
├── 08_preprocessor/  macros and conditional compilation
├── 09_modular/       a multi-file project: include/, src/, Makefile
└── 10_advanced/      function pointers, linked list, debugging
```

> **Tip:** Colab deletes its files when the runtime resets. If you come back later, run the **Setup** cell again and then the cells of the lesson you want.

## 0. Setup

Run this first. It creates the folders and checks that GCC is available.

In [1]:
!cd /content && mkdir -p c_course && cd c_course && mkdir -p 01_basics 02_control 03_functions 04_arrays 05_pointers 06_structs 07_files 08_preprocessor 09_modular/include 09_modular/src 10_advanced
!gcc --version | head -n 1
!echo "Folders ready:" && ls /content/c_course

gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
Folders ready:
01_basics   03_functions  05_pointers  07_files		09_modular
02_control  04_arrays	  06_structs   08_preprocessor	10_advanced


---
# Part 1 — Basics

## 1.1 Your first program

Every C program has the same skeleton:

- `#include <stdio.h>` pulls in the **standard input/output** library, which gives us `printf`.
- `int main(void)` is where execution starts. Exactly one `main` per program.
- Statements end with `;` and blocks are wrapped in `{ }`.
- `return 0;` tells the operating system the program succeeded.

C is a **compiled** language: the source file is turned into machine code first, and only then can it run.

In [ ]:
%%writefile /content/c_course/01_basics/hello.c
// hello.c - your first C program
#include <stdio.h>   // gives us printf

int main(void)       // every C program starts running at main
{
    printf("Hello, C!\n");   // \n means "new line"
    return 0;                // 0 = success
}

In [ ]:
!cd /content/c_course/01_basics && gcc -Wall -Wextra -std=c11 hello.c -o hello && ./hello

**What happened:** `gcc` read `hello.c`, produced an executable called `hello`, and `&&` ran it only because compiling succeeded. Try deleting a `;` from the code, re-run both cells, and read the error message: learning to read compiler errors is a core C skill.

## 1.2 Variables and data types

C is **statically typed**: every variable has a fixed type chosen when you declare it.

| Type | Typical size | Holds | `printf` format |
|---|---|---|---|
| `char` | 1 byte | a single character (really a small integer) | `%c` or `%d` |
| `int` | 4 bytes | whole numbers | `%d` |
| `unsigned int` | 4 bytes | non-negative whole numbers | `%u` |
| `long` | 8 bytes (on 64-bit Linux) | bigger whole numbers | `%ld` |
| `float` | 4 bytes | decimals, ~7 digits precision | `%f` |
| `double` | 8 bytes | decimals, ~15 digits precision | `%f` / `%lf` |
| `bool` | 1 byte | `true` / `false` (needs `<stdbool.h>`) | `%d` |

Sizes can differ between machines, which is why `sizeof` exists. On microcontrollers an `int` is often only 2 bytes; that's one reason embedded code uses `stdint.h` types (see the embedded notebooks).

`const` makes a variable read-only after it is initialised.

In [ ]:
%%writefile /content/c_course/01_basics/types.c
#include <stdio.h>
#include <limits.h>   // INT_MIN, INT_MAX ...
#include <float.h>    // FLT_DIG ...
#include <stdbool.h>  // bool, true, false

int main(void)
{
    char   letter = 'A';
    int    age    = 21;
    long   big    = 1234567890L;
    float  pi_f   = 3.14159f;
    double pi_d   = 3.141592653589793;
    bool   ready  = true;
    unsigned int count = 42u;

    printf("char    : %c (stored as the number %d)\n", letter, letter);
    printf("int     : %d\n", age);
    printf("long    : %ld\n", big);
    printf("float   : %.5f\n", pi_f);
    printf("double  : %.15f\n", pi_d);
    printf("bool    : %d\n", ready);
    printf("unsigned: %u\n\n", count);

    printf("Sizes on this machine (bytes):\n");
    printf("char=%zu short=%zu int=%zu long=%zu long long=%zu\n",
           sizeof(char), sizeof(short), sizeof(int), sizeof(long), sizeof(long long));
    printf("float=%zu double=%zu pointer=%zu\n\n",
           sizeof(float), sizeof(double), sizeof(void *));

    printf("int range: %d to %d\n", INT_MIN, INT_MAX);
    printf("float precision: about %d digits\n", FLT_DIG);

    const int DAYS_IN_WEEK = 7;   // const: cannot be changed later
    printf("Days in week: %d\n", DAYS_IN_WEEK);
    return 0;
}

In [ ]:
!cd /content/c_course/01_basics && gcc -Wall -Wextra -std=c11 types.c -o types && ./types

## 1.3 Operators

- **Arithmetic:** `+ - * / %`. Dividing two integers throws away the fraction (`17 / 5` is `3`). Cast to `double` when you want decimals.
- **Increment:** `x++` uses the value and then adds 1; `++x` adds 1 first.
- **Comparison:** `== != < > <= >=` give `1` (true) or `0` (false). Watch out: `=` assigns, `==` compares.
- **Logical:** `&&` (and), `||` (or), `!` (not).
- **Ternary:** `condition ? a : b` is a compact if/else expression.
- **Compound assignment:** `+= -= *= /=`.

In [ ]:
%%writefile /content/c_course/01_basics/operators.c
#include <stdio.h>

int main(void)
{
    int a = 17, b = 5;

    printf("a + b = %d\n", a + b);
    printf("a - b = %d\n", a - b);
    printf("a * b = %d\n", a * b);
    printf("a / b = %d   (integer division drops the fraction)\n", a / b);
    printf("a %% b = %d   (remainder)\n", a % b);   // %% inside printf prints one %
    printf("(double)a / b = %.2f\n\n", (double)a / b);

    int x = 5;
    printf("x++ gives %d, ", x++);    // use, then increment
    printf("now x = %d\n", x);
    printf("++x gives %d\n\n", ++x);  // increment, then use

    printf("a > b  -> %d\n", a > b);
    printf("a == b -> %d\n", a == b);
    printf("(a > 10 && b > 10) -> %d\n", a > 10 && b > 10);
    printf("(a > 10 || b > 10) -> %d\n", a > 10 || b > 10);
    printf("!(a > b) -> %d\n\n", !(a > b));

    int bigger = (a > b) ? a : b;     // ternary operator
    printf("bigger = %d\n", bigger);

    int total = 10;
    total += 5;    // 15
    total *= 2;    // 30
    total -= 3;    // 27
    printf("total = %d\n", total);
    return 0;
}

In [ ]:
!cd /content/c_course/01_basics && gcc -Wall -Wextra -std=c11 operators.c -o operators && ./operators

## 1.4 Reading input with `scanf`

`scanf` reads typed input. You pass the **address** of each variable (`&a`) so `scanf` can store into it; you'll understand why in the pointers lesson.

Colab cells can't easily type into a running C program, so we **pipe** input in with `echo "12 30" | ./input`. Change the numbers and re-run.

Always check `scanf`'s return value: it tells you how many items were read successfully.

In [ ]:
%%writefile /content/c_course/01_basics/input.c
#include <stdio.h>

int main(void)
{
    int a, b;
    printf("Enter two numbers: ");
    if (scanf("%d %d", &a, &b) != 2) {   // returns how many values it read
        printf("Invalid input\n");
        return 1;
    }
    printf("\n%d + %d = %d\n", a, b, a + b);
    return 0;
}

In [ ]:
!cd /content/c_course/01_basics && gcc -Wall -Wextra -std=c11 input.c -o input && echo "12 30" | ./input && echo "hello" | ./input

---
# Part 2 — Control Flow

## 2.1 `if / else` and `switch`

- `if / else if / else` chooses between branches using any condition.
- `switch` compares one integer (or `char`) value against fixed `case` labels. Each case needs a `break`, or execution "falls through" into the next case. `default` catches everything else.

(The loop and array used in `main` are covered properly in the next lessons.)

In [ ]:
%%writefile /content/c_course/02_control/grade.c
#include <stdio.h>

char grade_for(int score)
{
    if (score >= 90)      return 'A';
    else if (score >= 75) return 'B';
    else if (score >= 60) return 'C';
    else                  return 'F';
}

int main(void)
{
    int scores[] = {95, 80, 62, 40};

    for (int i = 0; i < 4; i++) {
        char g = grade_for(scores[i]);
        printf("Score %3d -> Grade %c -> ", scores[i], g);

        switch (g) {
            case 'A': printf("Excellent!\n"); break;
            case 'B': printf("Good job\n");   break;
            case 'C': printf("Passed\n");     break;
            default:  printf("Try again\n");  break;
        }
    }
    return 0;
}

In [ ]:
!cd /content/c_course/02_control && gcc -Wall -Wextra -std=c11 grade.c -o grade && ./grade

## 2.2 Loops

| Loop | Use when |
|---|---|
| `for (init; condition; step)` | you know how many times to repeat |
| `while (condition)` | you repeat until something changes |
| `do { } while (condition);` | the body must run at least once |

`break` exits a loop immediately; `continue` skips to the next pass. `for (;;)` is an infinite loop, extremely common in embedded programs (the "super loop").

In [ ]:
%%writefile /content/c_course/02_control/loops.c
#include <stdio.h>

int main(void)
{
    printf("for loop: ");
    for (int i = 1; i <= 5; i++) printf("%d ", i);
    printf("\n");

    printf("while loop (countdown): ");
    int n = 5;
    while (n > 0) {
        printf("%d ", n);
        n--;
    }
    printf("\n");

    printf("do-while runs at least once: ");
    int k = 100;
    do {
        printf("k=%d ", k);
    } while (k < 10);       // false, but the body already ran once
    printf("\n");

    printf("continue skips evens: ");
    for (int i = 1; i <= 10; i++) {
        if (i % 2 == 0) continue;
        printf("%d ", i);
    }
    printf("\n");

    printf("break at the first multiple of 7 after 20: ");
    for (int i = 21; ; i++) {          // no condition = runs forever...
        if (i % 7 == 0) {
            printf("%d\n", i);
            break;                     // ...until break
        }
    }

    printf("\nNested loops - multiplication table:\n");
    for (int r = 1; r <= 3; r++) {
        for (int c = 1; c <= 5; c++) printf("%4d", r * c);
        printf("\n");
    }
    return 0;
}

In [ ]:
!cd /content/c_course/02_control && gcc -Wall -Wextra -std=c11 loops.c -o loops && ./loops

---
# Part 3 — Functions

A function packages code under a name so it can be reused.

- A **prototype** (`int add(int a, int b);`) declares a function before it's used; the body can come later or live in another file.
- C passes arguments **by value**: the function gets a copy, so changing a parameter doesn't affect the caller. To modify the caller's variable, pass a pointer (Part 5).
- **Recursion**: a function may call itself. It always needs a base case that stops it.
- **Scope:** variables declared inside `{ }` exist only there. **Global** variables live for the whole program.
- A `static` **local** variable keeps its value between calls.

In [ ]:
%%writefile /content/c_course/03_functions/functions.c
#include <stdio.h>

/* Prototypes: tell the compiler which functions exist */
int  add(int a, int b);
void try_to_change(int x);
long factorial(int n);
void counter(void);

int global_total = 0;   /* global: visible to every function in this file */

int main(void)
{
    printf("add(3, 4) = %d\n\n", add(3, 4));

    int value = 10;
    try_to_change(value);
    printf("back in main, value is still %d (pass by value)\n\n", value);

    for (int i = 0; i <= 5; i++)
        printf("%d! = %ld\n", i, factorial(i));
    printf("\n");

    counter();
    counter();
    counter();
    printf("global_total = %d\n", global_total);
    return 0;
}

int add(int a, int b)
{
    return a + b;
}

void try_to_change(int x)
{
    x = 999;            /* only changes the local copy */
    printf("inside try_to_change, x = %d\n", x);
}

long factorial(int n)   /* recursion: a function that calls itself */
{
    if (n <= 1) return 1;            /* base case */
    return n * factorial(n - 1);     /* recursive case */
}

void counter(void)
{
    static int calls = 0;   /* static local: remembers its value between calls */
    calls++;
    global_total += 10;
    printf("counter() called %d time(s)\n", calls);
}

In [ ]:
!cd /content/c_course/03_functions && gcc -Wall -Wextra -std=c11 functions.c -o functions && ./functions

---
# Part 4 — Arrays and Strings

## 4.1 Arrays

An array is a fixed-size block of elements of the same type, stored next to each other in memory.

- Indexes start at **0**: an array of 5 has indexes 0 to 4.
- C does **not** check bounds. Reading `marks[10]` on a 5-element array is a bug the compiler usually won't catch (Part 10 shows a tool that does).
- When passed to a function, an array is not copied; the function works on the original. That's why you also pass its length.
- `sizeof(arr) / sizeof(arr[0])` gives the element count, but only where the array was declared.

In [ ]:
%%writefile /content/c_course/04_arrays/arrays.c
#include <stdio.h>

#define SIZE 5

double average(const int arr[], int n)   /* const: promise not to modify */
{
    int sum = 0;
    for (int i = 0; i < n; i++) sum += arr[i];
    return (double)sum / n;
}

void double_all(int arr[], int n)        /* changes the caller's array */
{
    for (int i = 0; i < n; i++) arr[i] *= 2;
}

int main(void)
{
    int marks[SIZE] = {70, 85, 90, 60, 95};

    printf("marks: ");
    for (int i = 0; i < SIZE; i++) printf("%d ", marks[i]);
    printf("\naverage = %.2f\n", average(marks, SIZE));

    double_all(marks, SIZE);
    printf("after double_all: ");
    for (int i = 0; i < SIZE; i++) printf("%d ", marks[i]);
    printf("\n");

    int count = sizeof(marks) / sizeof(marks[0]);
    printf("element count via sizeof = %d\n\n", count);

    int matrix[2][3] = { {1, 2, 3},
                         {4, 5, 6} };
    printf("2D array:\n");
    for (int r = 0; r < 2; r++) {
        for (int c = 0; c < 3; c++) printf("%3d", matrix[r][c]);
        printf("\n");
    }
    return 0;
}

In [ ]:
!cd /content/c_course/04_arrays && gcc -Wall -Wextra -std=c11 arrays.c -o arrays && ./arrays

## 4.2 Strings

C has no separate string type. A string is a `char` array that ends with the **null terminator** `'\0'`. Functions like `strlen` and `printf("%s")` walk the array until they hit it.

Useful functions from `<string.h>`: `strlen`, `strcmp`, `strncpy`, `strncat`. Prefer the "n" versions and `snprintf`, which take the buffer size and can't write past the end. Plain `strcpy`, `strcat`, and especially `gets` are a classic source of buffer overflows.

In [ ]:
%%writefile /content/c_course/04_arrays/strings.c
#include <stdio.h>
#include <string.h>
#include <ctype.h>    // toupper

int main(void)
{
    char name[20] = "Embedded";   /* a string is a char array ending in '\0' */
    char other[]  = "C";

    printf("name = %s, strlen = %zu, array size = %zu\n",
           name, strlen(name), sizeof(name));

    char full[40];
    snprintf(full, sizeof(full), "%s %s", name, other);   /* safe: never overflows */
    printf("full = \"%s\"\n", full);

    strncat(full, " rocks", sizeof(full) - strlen(full) - 1);   /* safe append */
    printf("after strncat = \"%s\"\n", full);

    printf("strcmp(\"abc\", \"abd\") = %d  (negative: first comes before second)\n",
           strcmp("abc", "abd"));

    for (size_t i = 0; full[i] != '\0'; i++)
        full[i] = (char)toupper((unsigned char)full[i]);
    printf("uppercase = %s\n", full);

    printf("bytes of \"%s\": ", other);
    for (size_t i = 0; i <= strlen(other); i++)
        printf("[%d] ", other[i]);
    printf("<- the 0 is the terminator\n");
    return 0;
}

In [ ]:
!cd /content/c_course/04_arrays && gcc -Wall -Wextra -std=c11 strings.c -o strings && ./strings

---
# Part 5 — Pointers and Memory

## 5.1 Pointers

A **pointer** is a variable that holds a memory address. This is the idea that makes C powerful, and the one that's essential for embedded work, where hardware registers are just addresses.

| Syntax | Meaning |
|---|---|
| `int *p;` | `p` is a pointer to an `int` |
| `&x` | "address of `x`" |
| `*p` | "the value at the address in `p`" (dereference) |
| `p + 1` | the next element: moves by `sizeof(*p)` bytes, not by 1 byte |
| `NULL` | "points to nothing"; never dereference it |

Pointers let a function change the caller's variables (`swap` below), and an array name automatically turns into a pointer to its first element.

`const int *p` means you can't change the value through `p`; `int *const p` means `p` itself can't point elsewhere.

In [ ]:
%%writefile /content/c_course/05_pointers/pointers.c
#include <stdio.h>

void swap(int *a, int *b)   /* receives addresses, so it can change the originals */
{
    int temp = *a;
    *a = *b;
    *b = temp;
}

int main(void)
{
    int x = 42;
    int *p = &x;            /* p holds the address of x */

    printf("x = %d\n", x);
    printf("address of x  = %p\n", (void *)&x);
    printf("p contains    = %p\n", (void *)p);
    printf("*p (value there) = %d\n", *p);

    *p = 100;               /* change x through the pointer */
    printf("after *p = 100, x = %d\n\n", x);

    int a = 1, b = 2;
    swap(&a, &b);
    printf("after swap: a = %d, b = %d\n\n", a, b);

    int nums[] = {10, 20, 30, 40};
    int *q = nums;          /* array name -> pointer to first element */
    for (int i = 0; i < 4; i++)
        printf("*(q + %d) = %d   at %p\n", i, *(q + i), (void *)(q + i));
    printf("addresses step by sizeof(int) = %zu bytes\n\n", sizeof(int));

    const int *read_only = &x;    /* can't change *read_only */
    int *const fixed = &x;        /* can't change where fixed points */
    *fixed = 7;
    printf("x seen through read_only = %d\n", *read_only);

    int *nothing = NULL;          /* always initialise pointers */
    if (nothing == NULL) printf("nothing is NULL, so we don't dereference it\n");

    int **pp = &p;                /* a pointer to a pointer */
    printf("**pp = %d\n", **pp);
    return 0;
}

In [ ]:
!cd /content/c_course/05_pointers && gcc -Wall -Wextra -std=c11 pointers.c -o pointers && ./pointers

## 5.2 Dynamic memory (the heap)

Local variables live on the **stack** and disappear when their function returns. When you need memory whose size is only known at runtime, or that must outlive a function, ask the **heap**:

- `malloc(bytes)` gives uninitialised memory; `calloc(count, size)` gives zeroed memory.
- `realloc(ptr, new_bytes)` grows or shrinks a block (it may move it).
- `free(ptr)` gives it back. Every `malloc` needs exactly one `free`, or you have a **memory leak**.
- All of them return `NULL` on failure. Always check.

Writing `malloc(n * sizeof *arr)` instead of `sizeof(int)` keeps the code correct if the type ever changes. On small microcontrollers the heap is often avoided altogether; see the memory-pool lesson in the embedded notebooks.

In [ ]:
%%writefile /content/c_course/05_pointers/dynamic.c
#include <stdio.h>
#include <stdlib.h>   // malloc, calloc, realloc, free

int main(void)
{
    int n = 5;
    int *arr = malloc(n * sizeof *arr);        /* ask the heap for 5 ints */
    if (arr == NULL) { printf("malloc failed\n"); return 1; }

    for (int i = 0; i < n; i++) arr[i] = (i + 1) * 10;
    printf("malloc'd array: ");
    for (int i = 0; i < n; i++) printf("%d ", arr[i]);
    printf("\n");

    int *bigger = realloc(arr, 2 * n * sizeof *arr);   /* grow to 10 ints */
    if (bigger == NULL) { free(arr); return 1; }        /* old block still valid on failure */
    arr = bigger;
    for (int i = n; i < 2 * n; i++) arr[i] = (i + 1) * 10;
    n *= 2;
    printf("after realloc:  ");
    for (int i = 0; i < n; i++) printf("%d ", arr[i]);
    printf("\n");

    free(arr);      /* give the memory back */
    arr = NULL;     /* avoid a dangling pointer */

    double *zeros = calloc(4, sizeof *zeros);   /* calloc zero-fills */
    if (zeros == NULL) return 1;
    printf("calloc'd values: %.1f %.1f %.1f %.1f\n", zeros[0], zeros[1], zeros[2], zeros[3]);
    free(zeros);
    return 0;
}

In [ ]:
!cd /content/c_course/05_pointers && gcc -Wall -Wextra -std=c11 dynamic.c -o dynamic && ./dynamic

---
# Part 6 — Structs, Unions, Enums, typedef

- A **struct** groups related variables of different types into one record. Access members with `.`, or with `->` when you have a pointer to the struct.
- Pass large structs to functions **by pointer** (`const Student *s`) to avoid copying them.
- An **enum** gives names to integer constants, which makes code readable (`GREEN` instead of `1`).
- A **union** stores all its members in the **same** memory, so it's as big as its largest member. It's useful for viewing the same bytes in different ways, as below.
- **typedef** creates a short alias for a type, so you can write `Student` instead of `struct Student`.

In [ ]:
%%writefile /content/c_course/06_structs/structs.c
#include <stdio.h>
#include <string.h>

typedef enum { RED, GREEN, BLUE } Color;   /* RED = 0, GREEN = 1, BLUE = 2 */

typedef struct {
    int x;
    int y;
} Point;

typedef struct {
    char  name[32];
    int   roll;
    float cgpa;
    Point desk;         /* a struct inside a struct */
} Student;

typedef union {         /* all members share the same 4 bytes */
    unsigned int  whole;
    unsigned char bytes[4];
} Word;

void print_student(const Student *s)    /* pointer: no copy is made */
{
    printf("%-8s roll=%d cgpa=%.1f desk=(%d,%d)\n",
           s->name, s->roll, s->cgpa, s->desk.x, s->desk.y);
}

void promote(Student *s)
{
    s->cgpa += 0.5f;    /* -> because s is a pointer */
}

int main(void)
{
    Student a = { "Ravi", 1, 8.2f, {1, 3} };

    Student b;
    strcpy(b.name, "Anita");
    b.roll = 2;
    b.cgpa = 9.1f;
    b.desk = (Point){2, 4};

    print_student(&a);
    print_student(&b);
    promote(&a);
    printf("after promote: ");
    print_student(&a);

    Student class[] = { a, b, { "Kabir", 3, 7.5f, {3, 1} } };
    printf("\nclass list:\n");
    for (size_t i = 0; i < sizeof class / sizeof class[0]; i++)
        print_student(&class[i]);

    Color c = GREEN;
    const char *names[] = { "RED", "GREEN", "BLUE" };
    printf("\nenum: c = %d (%s)\n", c, names[c]);

    Word w;
    w.whole = 0x11223344;
    printf("union: whole = 0x%08X, bytes = %02X %02X %02X %02X\n",
           w.whole, w.bytes[0], w.bytes[1], w.bytes[2], w.bytes[3]);
    printf("sizeof(Student) = %zu, sizeof(Word) = %zu\n", sizeof(Student), sizeof(Word));
    return 0;
}

In [ ]:
!cd /content/c_course/06_structs && gcc -Wall -Wextra -std=c11 structs.c -o structs && ./structs

Notice the union printed the bytes in **reverse** order (`44 33 22 11`). That's because Colab's x86 processor is **little-endian**: it stores the lowest byte first. Endianness matters a lot when talking to sensors and networks; the first embedded notebook covers it.

---
# Part 7 — File Input / Output

Files are handled through a `FILE *` pointer:

| Function | Purpose |
|---|---|
| `fopen(name, mode)` | open: `"r"` read, `"w"` write (erases), `"a"` append |
| `fprintf` / `fputs` | write text |
| `fscanf` / `fgets` | read text |
| `fclose` | close (flushes data to disk) |

`fopen` returns `NULL` if it fails, and `perror` prints the reason.

In [ ]:
%%writefile /content/c_course/07_files/files.c
#include <stdio.h>

int main(void)
{
    FILE *fp = fopen("students.txt", "w");   /* "w" creates or overwrites */
    if (fp == NULL) { perror("fopen"); return 1; }
    fprintf(fp, "Ravi 82\n");
    fprintf(fp, "Anita 91\n");
    fprintf(fp, "Kabir 75\n");
    fclose(fp);
    printf("Wrote students.txt\n");

    fp = fopen("students.txt", "r");
    if (fp == NULL) { perror("fopen"); return 1; }

    char name[32];
    int marks, total = 0, count = 0;
    while (fscanf(fp, "%31s %d", name, &marks) == 2) {   /* %31s: never overflow name */
        printf("read: %-6s %d\n", name, marks);
        total += marks;
        count++;
    }
    fclose(fp);
    if (count > 0) printf("average = %.2f\n", (double)total / count);

    fp = fopen("students.txt", "a");    /* append to the end */
    if (fp) {
        fputs("Meera 88\n", fp);
        fclose(fp);
    }
    return 0;
}

In [ ]:
!cd /content/c_course/07_files && gcc -Wall -Wextra -std=c11 files.c -o files && ./files && echo "--- students.txt now contains ---" && cat students.txt

---
# Part 8 — The Preprocessor

Before compiling, the **preprocessor** rewrites your source text. Its directives start with `#`.

- `#include` pastes in another file.
- `#define NAME value` makes a constant; `#define F(x) ...` makes a macro that looks like a function but is plain text substitution. Always wrap macro parameters and the whole body in parentheses, as `SQUARE_BAD` below shows why.
- `#ifdef / #if / #else / #endif` include or skip code at compile time. You can set values from the command line with `-DNAME` or `-DNAME=value`, which is how one codebase builds debug and release versions, or targets different boards.
- Built-in macros like `__FILE__` and `__LINE__` help with logging.

In [ ]:
%%writefile /content/c_course/08_preprocessor/macros.c
#include <stdio.h>

#define PI 3.14159
#define SQUARE_BAD(x) x * x              /* no parentheses: dangerous */
#define SQUARE(x)     ((x) * (x))        /* always parenthesise */
#define MAX(a, b)     ((a) > (b) ? (a) : (b))
#define ARRAY_LEN(arr) (sizeof(arr) / sizeof((arr)[0]))
#define STR(x) #x                        /* # turns code into a string */

#ifndef MODE
#define MODE 1                           /* default when not set with -DMODE=... */
#endif

int main(void)
{
    printf("PI = %f\n", PI);
    printf("SQUARE_BAD(2 + 3) = %d   (expands to 2 + 3 * 2 + 3)\n", SQUARE_BAD(2 + 3));
    printf("SQUARE(2 + 3)     = %d\n", SQUARE(2 + 3));
    printf("MAX(4, 9) = %d\n", MAX(4, 9));

    int data[] = {1, 2, 3, 4, 5, 6};
    printf("ARRAY_LEN(data) = %zu\n", ARRAY_LEN(data));
    printf("STR(hello world) = \"%s\"\n", STR(hello world));
    printf("This is file %s, line %d\n", __FILE__, __LINE__);

#ifdef DEBUG
    printf("[DEBUG] debug build: extra logging is on\n");
#endif

#if MODE == 1
    printf("MODE 1: normal\n");
#elif MODE == 2
    printf("MODE 2: fast\n");
#else
    printf("Unknown MODE\n");
#endif
    return 0;
}

In [ ]:
!cd /content/c_course/08_preprocessor && echo "=== normal build ===" && gcc -Wall -Wextra -std=c11 macros.c -o macros && ./macros
!cd /content/c_course/08_preprocessor && echo "=== built with -DDEBUG -DMODE=2 ===" && gcc -Wall -Wextra -std=c11 -DDEBUG -DMODE=2 macros.c -o macros && ./macros

You can look at the code **after** preprocessing with `gcc -E`. Here's the end of `main`, with every macro already replaced:

In [ ]:
!cd /content/c_course/08_preprocessor && gcc -E -P macros.c | tail -n 16

---
# Part 9 — Modular Programming: Multi-File Projects

Real programs are split across many files. This is the most important organisational skill in C, and it's exactly how embedded firmware is structured (drivers, HAL, application).

## The rules

- A **header** (`.h`) holds the **interface**: type definitions, function prototypes, `extern` declarations. Other files `#include` it.
- A **source file** (`.c`) holds the **implementation**: function bodies and variable definitions.
- **Header guards** (`#ifndef X_H / #define X_H / #endif`) stop a header from being included twice.
- `static` on a global variable or function makes it **private** to its `.c` file. Use it for internal helpers.
- `extern int x;` in a header **declares** a global that is **defined** (given memory) in exactly one `.c` file.

## The project

```
09_modular/
├── include/
│   ├── student.h     Student type + functions (public interface)
│   └── utils.h       small helpers
├── src/
│   ├── student.c     implementation of student.h
│   ├── utils.c       implementation of utils.h (with private static helpers)
│   └── main.c        the application
└── Makefile          build instructions
```

In [ ]:
%%writefile /content/c_course/09_modular/include/student.h
#ifndef STUDENT_H          /* header guard: safe to include more than once */
#define STUDENT_H

#define NAME_LEN 32

typedef struct {
    char  name[NAME_LEN];
    int   roll;
    float marks;
} Student;

/* Public interface: what other files are allowed to use */
void           student_init(Student *s, const char *name, int roll, float marks);
void           student_print(const Student *s);
float          student_average(const Student list[], int count);
const Student *student_top(const Student list[], int count);

#endif /* STUDENT_H */

In [ ]:
%%writefile /content/c_course/09_modular/include/utils.h
#ifndef UTILS_H
#define UTILS_H

extern int students_created;   /* declaration only: the variable lives in utils.c */

float clamp_float(float value, float low, float high);
void  print_line(int width);
int   lines_drawn(void);

#endif /* UTILS_H */

In [ ]:
%%writefile /content/c_course/09_modular/src/utils.c
#include <stdio.h>
#include "utils.h"

int students_created = 0;        /* the one real definition of the extern variable */

static int lines_printed = 0;    /* static global: private to this file */

static void repeat_char(char c, int n)   /* static function: private helper */
{
    for (int i = 0; i < n; i++) putchar(c);
}

float clamp_float(float value, float low, float high)
{
    if (value < low)  return low;
    if (value > high) return high;
    return value;
}

void print_line(int width)
{
    repeat_char('-', width);
    putchar('\n');
    lines_printed++;
}

int lines_drawn(void)
{
    return lines_printed;        /* controlled read access to private data */
}

In [ ]:
%%writefile /content/c_course/09_modular/src/student.c
#include <stdio.h>
#include <string.h>
#include "student.h"
#include "utils.h"

void student_init(Student *s, const char *name, int roll, float marks)
{
    strncpy(s->name, name, NAME_LEN - 1);
    s->name[NAME_LEN - 1] = '\0';                 /* strncpy may not add the terminator */
    s->roll  = roll;
    s->marks = clamp_float(marks, 0.0f, 100.0f);  /* from utils.c */
    students_created++;                           /* extern variable from utils.c */
}

void student_print(const Student *s)
{
    printf("  #%-3d %-10s %6.1f\n", s->roll, s->name, s->marks);
}

float student_average(const Student list[], int count)
{
    if (count <= 0) return 0.0f;
    float sum = 0.0f;
    for (int i = 0; i < count; i++) sum += list[i].marks;
    return sum / count;
}

const Student *student_top(const Student list[], int count)
{
    if (count <= 0) return NULL;
    const Student *best = &list[0];
    for (int i = 1; i < count; i++)
        if (list[i].marks > best->marks) best = &list[i];
    return best;
}

In [ ]:
%%writefile /content/c_course/09_modular/src/main.c
#include <stdio.h>
#include "student.h"
#include "utils.h"

int main(void)
{
    Student class[3];
    student_init(&class[0], "Ravi",  1,  82.5f);
    student_init(&class[1], "Anita", 2,  91.0f);
    student_init(&class[2], "Kabir", 3, 105.0f);   /* gets clamped to 100 */

    print_line(26);
    printf("  Roll Name        Marks\n");
    print_line(26);
    for (int i = 0; i < 3; i++) student_print(&class[i]);
    print_line(26);

    printf("Average : %.2f\n", student_average(class, 3));
    const Student *top = student_top(class, 3);
    if (top) printf("Topper  : %s\n", top->name);
    printf("Students created: %d, lines drawn: %d\n", students_created, lines_drawn());
    return 0;
}

## Building it by hand: compile, then link

Building a multi-file program has two stages:

1. **Compile** each `.c` separately into an **object file** (`.o`) with `-c`. `-Iinclude` tells GCC where the headers are.
2. **Link** the object files together into one program. The linker connects calls like `clamp_float()` in `student.c` to the code in `utils.o`.

`nm` lists the symbols in an object file. Uppercase `T` / `D` / `B` means public, lowercase `t` / `b` means `static` (private), and `U` means "used here, defined somewhere else".

In [ ]:
!cd /content/c_course/09_modular && mkdir -p build && gcc -Wall -Wextra -std=c11 -Iinclude -c src/utils.c   -o build/utils.o && gcc -Wall -Wextra -std=c11 -Iinclude -c src/student.c -o build/student.o && gcc -Wall -Wextra -std=c11 -Iinclude -c src/main.c    -o build/main.o && echo "object files:" && ls build && gcc build/main.o build/student.o build/utils.o -o build/school && echo "--- running ---" && ./build/school
!echo "--- symbols in utils.o ---" && nm /content/c_course/09_modular/build/utils.o

## Building with a Makefile

Typing those commands gets tedious fast. A **Makefile** describes how to build each file and what it depends on, and `make` rebuilds **only what changed**.

Each rule looks like:
```
target: dependencies
<TAB>command
```
The command line **must start with a real Tab character**, not spaces (the most common Makefile error). `$@` means the target, `$<` the first dependency.

In [ ]:
%%writefile /content/c_course/09_modular/Makefile
CC      = gcc
CFLAGS  = -Wall -Wextra -std=c11 -Iinclude
SRC     = $(wildcard src/*.c)
OBJ     = $(patsubst src/%.c,build/%.o,$(SRC))
TARGET  = build/school

all: $(TARGET)

$(TARGET): $(OBJ)
	$(CC) $(OBJ) -o $@

build/%.o: src/%.c include/*.h | build
	$(CC) $(CFLAGS) -c $< -o $@

build:
	mkdir -p build

run: $(TARGET)
	./$(TARGET)

clean:
	rm -rf build

.PHONY: all run clean

In [ ]:
!cd /content/c_course/09_modular && make clean && make && make run
!echo "=== touch main.c and rebuild: only main.o is recompiled ===" && cd /content/c_course/09_modular && touch src/main.c && make

---
# Part 10 — Advanced Topics

## 10.1 Function pointers and callbacks

A function's name is also its address, so you can store functions in variables, put them in arrays, and pass them to other functions.

- A **callback** is a function you hand to another function to be called later. The standard `qsort` takes a comparison callback.
- A **dispatch table** (array of function pointers) replaces long `switch` statements. Embedded systems use the same idea for interrupt vector tables and command handlers.

`typedef int (*BinaryOp)(int, int);` reads as "BinaryOp is a pointer to a function that takes two ints and returns an int".

In [ ]:
%%writefile /content/c_course/10_advanced/func_ptr.c
#include <stdio.h>
#include <stdlib.h>   // qsort

int add(int a, int b) { return a + b; }
int sub(int a, int b) { return a - b; }
int mul(int a, int b) { return a * b; }

typedef int (*BinaryOp)(int, int);   /* pointer to a function taking 2 ints */

int apply(BinaryOp op, int a, int b) /* op is a callback */
{
    return op(a, b);
}

int ascending(const void *p, const void *q)
{
    int a = *(const int *)p;
    int b = *(const int *)q;
    return (a > b) - (a < b);        /* -1, 0 or 1, without overflow risk */
}

int descending(const void *p, const void *q)
{
    return ascending(q, p);
}

int main(void)
{
    BinaryOp    table[]  = { add, sub, mul };   /* dispatch table */
    const char *symbol[] = { "+", "-", "*" };
    for (int i = 0; i < 3; i++)
        printf("8 %s 3 = %d\n", symbol[i], apply(table[i], 8, 3));

    int nums[] = { 42, 7, 19, 3, 88, 25 };
    size_t n = sizeof nums / sizeof nums[0];

    qsort(nums, n, sizeof nums[0], ascending);
    printf("\nascending : ");
    for (size_t i = 0; i < n; i++) printf("%d ", nums[i]);

    qsort(nums, n, sizeof nums[0], descending);
    printf("\ndescending: ");
    for (size_t i = 0; i < n; i++) printf("%d ", nums[i]);
    printf("\n");
    return 0;
}

In [ ]:
!cd /content/c_course/10_advanced && gcc -Wall -Wextra -std=c11 func_ptr.c -o func_ptr && ./func_ptr

## 10.2 A dynamic data structure: linked list

A linked list is a chain of nodes allocated on the heap, each pointing to the next. It brings together structs, pointers, and `malloc`/`free`.

The `Node **head` (pointer to a pointer) parameter lets a function change which node is first, without special cases for an empty list.

In [ ]:
%%writefile /content/c_course/10_advanced/linked_list.c
#include <stdio.h>
#include <stdlib.h>

typedef struct Node {
    int          value;
    struct Node *next;      /* the struct refers to its own type */
} Node;

static Node *new_node(int value)
{
    Node *n = malloc(sizeof *n);
    if (!n) { perror("malloc"); exit(1); }
    n->value = value;
    n->next  = NULL;
    return n;
}

void push_front(Node **head, int value)
{
    Node *n = new_node(value);
    n->next = *head;
    *head   = n;
}

void append(Node **head, int value)
{
    while (*head) head = &(*head)->next;   /* walk to the last next-pointer */
    *head = new_node(value);
}

void remove_value(Node **head, int value)
{
    while (*head) {
        if ((*head)->value == value) {
            Node *dead = *head;
            *head = dead->next;            /* unlink */
            free(dead);
            return;
        }
        head = &(*head)->next;
    }
}

void print_list(const Node *head)
{
    for (const Node *p = head; p; p = p->next) printf("%d -> ", p->value);
    printf("NULL\n");
}

void free_list(Node *head)
{
    while (head) {
        Node *next = head->next;
        free(head);
        head = next;
    }
}

int main(void)
{
    Node *list = NULL;      /* empty list */
    append(&list, 20);
    append(&list, 30);
    push_front(&list, 10);
    append(&list, 40);
    print_list(list);

    remove_value(&list, 30);
    printf("after removing 30: ");
    print_list(list);

    free_list(list);        /* no leaks */
    return 0;
}

In [ ]:
!cd /content/c_course/10_advanced && gcc -Wall -Wextra -std=c11 linked_list.c -o linked_list && ./linked_list

## 10.3 Finding memory bugs with AddressSanitizer

C trusts you completely, so bugs like writing past the end of an array often **seem to work** and then crash much later, or never in testing. GCC's **AddressSanitizer** (`-fsanitize=address -g`) adds checks that catch these on the spot and tell you the exact line.

The program below has an off-by-one bug. First we run it normally, then with the sanitizer.

In [ ]:
%%writefile /content/c_course/10_advanced/bug.c
#include <stdio.h>
#include <stdlib.h>

int main(void)
{
    int *data = malloc(5 * sizeof *data);
    if (!data) return 1;

    for (int i = 0; i <= 5; i++)     /* BUG: should be i < 5 */
        data[i] = i * 10;

    printf("data[0] = %d\n", data[0]);
    free(data);
    return 0;
}

In [ ]:
!cd /content/c_course/10_advanced && echo "=== normal build: looks fine ===" && gcc -Wall -Wextra -std=c11 bug.c -o bug && ./bug
!cd /content/c_course/10_advanced && echo "=== with AddressSanitizer ===" && gcc -Wall -Wextra -std=c11 -g -fsanitize=address bug.c -o bug_asan && ./bug_asan 2>&1 | head -n 12

Look for **heap-buffer-overflow** and the line number in `bug.c` in the report. Fix the loop to `i < 5`, re-run both cells, and the sanitizer goes quiet. Building with `-fsanitize=address,undefined` during development is a great habit.

---
# Exercises

Try these in new cells, using the same `%%writefile` + `!gcc` pattern:

1. **Basics:** read a temperature in Celsius (pipe it in with `echo`) and print it in Fahrenheit with 1 decimal place.
2. **Loops:** print all prime numbers below 100.
3. **Functions:** write a recursive `int fib(int n)` and print the first 15 Fibonacci numbers.
4. **Arrays:** find the second-largest value in an array without sorting it.
5. **Strings:** write your own `my_strlen` and `reverse_string` using pointers only, no `[]` indexing.
6. **Pointers:** write `void min_max(const int *arr, int n, int *min, int *max)` that returns two results through pointers.
7. **Structs + files:** save an array of `Student` structs to a file and load them back.
8. **Modular:** add `include/stats.h` and `src/stats.c` to the Part 9 project with a `student_count_above(list, n, cutoff)` function. The Makefile picks it up automatically.

When you're comfortable here, continue with the **Embedded C** notebooks.